# Detecção de Estados de Torneiras com YOLO  🚰

## Residência em Inteligência Artificial — Visão Computacional
### Desafio Semana 5 — Detecção de Estados com YOLO

---

Este notebook treina um modelo **YOLO** para detectar o **estado** de uma torneira:

| Classe | id | Significado |
|--------|----|-------------|
| `torneira_aberta`  | 0 | Torneira **aberta** (em uso / fluxo de água) |
| `torneira_fechada` | 1 | Torneira **fechada** (desligada) |

O dataset foi criado pelo grupo (fotos próprias), anotado no **Label Studio**
(formato YOLO) e contém **85 imagens** com variações de iluminação, ângulo,
distância e fundo.

### 🏭 Conexão com a indústria
Detectar se uma torneira/válvula está **aberta ou fechada** equivale a problemas
industriais reais: **monitoramento de válvulas/registros** em tubulações e plantas
(válvula aberta/fechada → segurança e controle de vazão), inspeção de atuadores em
linhas automatizadas, leitura de estado em painéis. Um erro de leitura de estado
(válvula que deveria estar fechada e está aberta) significa vazamento, desperdício
ou risco — por isso a detecção automática por visão computacional é tão útil.

---
### 📈 Nesta versão (v2) — melhorias sobre o baseline inicial
O primeiro treino deu `mAP50 ≈ 0.38` (melhor época = 4). Diagnóstico e correções:

1. **Augmentation calibrada para o problema** — desligamos `mosaic` e `mixup`, que
   são ótimos para COCO mas **péssimos aqui** (encolhem a torneira e misturam
   aberta+fechada, destruindo a pista de estado que é justamente o que classificamos).
2. **Validação cruzada 5-fold** — 17 imagens de validação dão uma métrica ruidosa
   (1 imagem ≈ 6% de mAP). Com 5-fold reportamos **média ± desvio**, bem mais confiável.
3. **Maior resolução** (`imgsz=768`) para preservar o detalhe do registro/alavanca.
4. **Diagnóstico por matriz de confusão** — separa erro de *localização* de erro de
   *estado*.
5. **Comparação de tamanho de modelo** (n/s/m) — mostra que aumentar o modelo (XL)
   **não** ajuda em 85 imagens.

> **Como usar:** `Ambiente de execução → Executar tudo` (use uma **GPU T4**).


## 1. Setup — instalação e checagem de ambiente

In [ ]:
# Instala a versão mais recente do Ultralytics (YOLO11)
%pip install -q "ultralytics>=8.3.0"

import ultralytics, torch
ultralytics.checks()
print("\nTorch:", torch.__version__, "| CUDA disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ Sem GPU — vá em 'Ambiente de execução → Alterar o tipo de ambiente → T4 GPU'.")


## 2. Configuração do experimento

Hiperparâmetros centralizados. Os defaults já estão calibrados para este dataset.

In [ ]:
from pathlib import Path

# --- Origem do dataset (repositório do grupo no GitHub) ---
REPO_URL    = "https://github.com/joaowinderfeldbussolotto/postgrad-cv-projects.git"
REPO_DIR    = "postgrad-cv-projects"
DATA_SUBDIR = "project5/dataset"        # pasta com images/, labels/, classes.txt

# --- Modelo base (transfer learning a partir do COCO) ---
# n=leve | s=recomendado | m=mais pesado.  NÃO use x/XL: em 85 imagens overfita
# (mais parâmetros precisam de muito mais dados). A Seção 8 comprova isso.
MODEL    = "yolo11s.pt"

# --- Hiperparâmetros de treino ---
IMGSZ    = 768        # 640 é mais rápido; 768 preserva o detalhe do registro/alavanca
EPOCHS   = 100
PATIENCE = 30         # early-stopping: para sozinho no melhor ponto
BATCH    = 16         # use -1 para batch automático conforme a VRAM
SEED     = 42
N_FOLDS  = 5          # validação cruzada estratificada

CLASS_NAMES = ["torneira_aberta", "torneira_fechada"]


## 3. Obter o dataset

Funciona em dois modos: clona o repositório (Colab) ou usa a pasta local (se já
estiver dentro do repo).

In [ ]:
import subprocess

def resolve_dataset_dir():
    for cand in [Path(DATA_SUBDIR), Path("..") / DATA_SUBDIR, Path("dataset")]:
        if (cand / "images").exists() and (cand / "labels").exists():
            return cand.resolve()
    cand = Path(REPO_DIR) / DATA_SUBDIR
    if (cand / "images").exists():
        return cand.resolve()
    print("Clonando o repositório do dataset...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    return (Path(REPO_DIR) / DATA_SUBDIR).resolve()

DATASET_DIR = resolve_dataset_dir()
IMAGES_DIR  = DATASET_DIR / "images"
LABELS_DIR  = DATASET_DIR / "labels"

n_imgs   = len(list(IMAGES_DIR.glob("*.*")))
n_labels = len(list(LABELS_DIR.glob("*.txt")))
print("Dataset:", DATASET_DIR)
print(f"Imagens: {n_imgs} | Labels: {n_labels}")
assert n_imgs > 0, "Nenhuma imagem encontrada!"


## 4. Exploração do dataset — distribuição das classes

In [ ]:
from collections import Counter

def label_path_for(img_path):
    return LABELS_DIR / (img_path.stem + ".txt")

images = sorted([p for p in IMAGES_DIR.glob("*.*")
                 if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}])

img_main_class = {}
class_box_counter = Counter()
for img in images:
    lp = label_path_for(img)
    cls_ids = []
    if lp.exists():
        for line in lp.read_text().strip().splitlines():
            if line.strip():
                c = int(float(line.split()[0])); cls_ids.append(c); class_box_counter[c] += 1
    img_main_class[img] = Counter(cls_ids).most_common(1)[0][0] if cls_ids else -1

print("Boxes por classe:")
for cid, name in enumerate(CLASS_NAMES):
    print(f"  {cid} {name}: {class_box_counter[cid]}")

import matplotlib.pyplot as plt
plt.figure(figsize=(5,3))
plt.bar([CLASS_NAMES[c] for c in sorted(class_box_counter)],
        [class_box_counter[c] for c in sorted(class_box_counter)],
        color=["#2a9d8f", "#e76f51"])
plt.title("Distribuição de classes (nº de bounding boxes)")
plt.ylabel("quantidade"); plt.tight_layout(); plt.show()


In [ ]:
# Visualiza algumas imagens com suas bounding boxes
import cv2, random

def draw_boxes(img_path):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    lp = label_path_for(img_path)
    colors = [(42,157,143), (231,111,81)]
    if lp.exists():
        for line in lp.read_text().strip().splitlines():
            if not line.strip(): continue
            c, xc, yc, bw, bh = map(float, line.split()); c = int(c)
            x1, y1 = int((xc-bw/2)*w), int((yc-bh/2)*h)
            x2, y2 = int((xc+bw/2)*w), int((yc+bh/2)*h)
            cv2.rectangle(img, (x1,y1), (x2,y2), colors[c], 3)
            cv2.putText(img, CLASS_NAMES[c], (x1, max(20,y1-8)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, colors[c], 2)
    return img

random.seed(SEED)
sample = random.sample(images, min(6, len(images)))
plt.figure(figsize=(14, 8))
for i, img in enumerate(sample):
    plt.subplot(2, 3, i+1); plt.imshow(draw_boxes(img)); plt.axis("off")
    plt.title(img.name[:22], fontsize=8)
plt.tight_layout(); plt.show()


## 5. Helper: montar um dataset YOLO a partir de uma lista de imagens

Função reutilizada pelo **baseline**, pela **validação cruzada** e pela
**comparação de modelos**. Copia imagens/labels para a estrutura esperada pelo
YOLO e escreve o `data.yaml`.

> **Observação de honestidade científica:** as fotos foram tiradas por vários
> integrantes (`s01`…`s10`). Um split *por integrante* daria métricas ainda mais
> conservadoras. Usamos split estratificado por classe (padrão do desafio).

In [ ]:
import shutil, yaml

def build_yolo_dataset(train_imgs, val_imgs, out_dir):
    """Cria out_dir/{images,labels}/{train,val} + data.yaml e retorna o caminho do yaml."""
    out_dir = Path(out_dir).resolve()
    if out_dir.exists():
        shutil.rmtree(out_dir)
    for split in ["train", "val"]:
        (out_dir / "images" / split).mkdir(parents=True, exist_ok=True)
        (out_dir / "labels" / split).mkdir(parents=True, exist_ok=True)

    def populate(img_list, split):
        for img in img_list:
            shutil.copy(img, out_dir / "images" / split / img.name)
            lp = label_path_for(img)
            dst = out_dir / "labels" / split / (img.stem + ".txt")
            dst.write_text(lp.read_text() if lp.exists() else "")

    populate(train_imgs, "train")
    populate(val_imgs, "val")

    data_yaml = {"path": str(out_dir), "train": "images/train", "val": "images/val",
                 "names": {i: n for i, n in enumerate(CLASS_NAMES)}}
    yaml_path = out_dir / "data.yaml"
    yaml_path.write_text(yaml.safe_dump(data_yaml, sort_keys=False, allow_unicode=True))
    return yaml_path


def stratified_split(img_list, val_frac=0.2, seed=SEED):
    """Split estratificado por classe dominante."""
    from collections import defaultdict
    rng = random.Random(seed)
    by_class = defaultdict(list)
    for img in img_list:
        by_class[img_main_class[img]].append(img)
    train, val = [], []
    for _, imgs in by_class.items():
        imgs = imgs[:]; rng.shuffle(imgs)
        n_val = max(1, round(len(imgs) * val_frac))
        val += imgs[:n_val]; train += imgs[n_val:]
    rng.shuffle(train); rng.shuffle(val)
    return train, val

print("Helpers prontos: build_yolo_dataset(), stratified_split()")


## 6. Baseline rápido (2 épocas)

Referência "antes" exigida pelo enunciado: poucas épocas, métrica baixa esperada.

In [ ]:
from ultralytics import YOLO

bl_train, bl_val = stratified_split(images, val_frac=0.2)
bl_yaml = build_yolo_dataset(bl_train, bl_val, "ds_baseline")
print(f"Baseline split → treino {len(bl_train)} | val {len(bl_val)}")

baseline = YOLO(MODEL)
baseline.train(data=str(bl_yaml), epochs=2, imgsz=IMGSZ, batch=BATCH, seed=SEED,
               project="runs_torneiras", name="baseline", exist_ok=True, verbose=False)
bm = baseline.val(data=str(bl_yaml), split="val", verbose=False)
BASELINE_MAP50 = float(bm.box.map50)
print(f"\nBASELINE  mAP50: {bm.box.map50:.3f} | mAP50-95: {bm.box.map:.3f}")


## 7. Treino com validação cruzada 5-fold (resultado principal)

### Por que esta é a etapa que corrige o problema

O baseline inicial atingia o melhor resultado já na **época 4** e depois não
melhorava — sinal de que a **augmentation estava atrapalhando**. Ajustes:

| Param | Antes | Agora | Motivo |
|-------|-------|-------|--------|
| `mosaic` | 1.0 | **0.0** | mosaic junta 4 imagens e encolhe a torneira → a pista de estado some |
| `mixup` | 0.1 | **0.0** | mixup mistura aberta+fechada → ruído de rótulo no que classificamos |
| `erasing` | 0.4 | **0.0** | não apagar a alavanca/registro (a pista visual) |
| `scale` | 0.5 | **0.3** | zoom mais suave |
| `degrees` | 10 | **8** | rotação leve (torneira é quase sempre na vertical) |
| `hsv_s` | 0.7 | **0.5** | jitter de cor mais moderado |
| `fliplr`,`translate` | 0.5, 0.1 | mantidos | flip horizontal e translação leve ajudam |

Treinamos `N_FOLDS` vezes em splits estratificados diferentes e reportamos
**média ± desvio** — métrica muito mais confiável que um único split de 17 imagens.

In [ ]:
import numpy as np
from sklearn.model_selection import StratifiedKFold

# Config de augmentation calibrada para "1 objeto centralizado, pista de estado sutil"
AUG = dict(
    hsv_h=0.015, hsv_s=0.5, hsv_v=0.4,
    degrees=8.0, translate=0.1, scale=0.3, fliplr=0.5, flipud=0.0,
    mosaic=0.0, mixup=0.0, erasing=0.0,
)

X = np.array(images, dtype=object)
y = np.array([img_main_class[i] for i in images])
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

fold_rows = []
best_fold = {"map50": -1.0, "weights": None, "save_dir": None, "yaml": None}

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
    tr_imgs = list(X[tr_idx]); va_imgs = list(X[va_idx])
    yaml_path = build_yolo_dataset(tr_imgs, va_imgs, f"ds_fold{fold}")
    print(f"\n========== FOLD {fold+1}/{N_FOLDS} "
          f"(treino {len(tr_imgs)} | val {len(va_imgs)}) ==========")
    m = YOLO(MODEL)
    m.train(data=str(yaml_path), epochs=EPOCHS, patience=PATIENCE, imgsz=IMGSZ,
            batch=BATCH, seed=SEED, optimizer="auto", cos_lr=True, cache=True,
            project="runs_torneiras", name=f"fold{fold}", exist_ok=True,
            verbose=False, plots=True, **AUG)
    res = m.val(data=str(yaml_path), split="val", verbose=False)
    row = {"fold": fold, "mAP50": float(res.box.map50), "mAP50-95": float(res.box.map),
           "P": float(res.box.mp), "R": float(res.box.mr)}
    fold_rows.append(row)
    print(f"Fold {fold+1}: mAP50={row['mAP50']:.3f} | mAP50-95={row['mAP50-95']:.3f}")
    if row["mAP50"] > best_fold["map50"]:
        sd = Path(m.trainer.save_dir)
        best_fold.update(map50=row["mAP50"], weights=sd / "weights" / "best.pt",
                         save_dir=sd, yaml=yaml_path)

BEST_WEIGHTS = best_fold["weights"]


In [ ]:
import pandas as pd
df = pd.DataFrame(fold_rows)
print(df.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

mean50, std50 = df["mAP50"].mean(), df["mAP50"].std()
mean5095, std5095 = df["mAP50-95"].mean(), df["mAP50-95"].std()
print("\n================ RESULTADO 5-FOLD (média ± desvio) ================")
print(f"mAP50    : {mean50:.3f} ± {std50:.3f}")
print(f"mAP50-95 : {mean5095:.3f} ± {std5095:.3f}")
print(f"\nBaseline (2 épocas) mAP50: {BASELINE_MAP50:.3f}  ->  "
      f"CV 5-fold mAP50: {mean50:.3f}  (melhor fold: {best_fold['map50']:.3f})")

plt.figure(figsize=(5,3))
plt.bar(df["fold"].astype(str), df["mAP50"], color="#2a9d8f")
plt.axhline(mean50, color="#e76f51", ls="--", label=f"média {mean50:.3f}")
plt.xlabel("fold"); plt.ylabel("mAP50"); plt.title("mAP50 por fold")
plt.legend(); plt.tight_layout(); plt.show()


## 8. Diagnóstico — matriz de confusão e curvas (melhor fold)

**Como ler a matriz de confusão:**
- Muita confusão **entre `torneira_aberta` e `torneira_fechada`** → problema de
  **estado**: o modelo acha a torneira mas erra se está aberta/fechada (pista sutil →
  precisa de fotos mais nítidas do registro / mais dados).
- Muita confusão com **`background`** (linha/coluna extra) → problema de
  **localização/detecção**: o modelo não encontra a torneira (ângulo/iluminação ruins).

In [ ]:
from IPython.display import Image, display

SAVE_DIR = best_fold["save_dir"]
print("Melhor fold:", SAVE_DIR)
for plot in ["confusion_matrix.png", "confusion_matrix_normalized.png",
             "BoxPR_curve.png", "BoxF1_curve.png", "results.png"]:
    p = SAVE_DIR / plot
    if p.exists():
        print(plot); display(Image(filename=str(p), width=560))


## 9. Comparação de tamanho de modelo (por que NÃO usar XL / yolo26)

Treino rápido de `yolo11n`, `yolo11s` e `yolo11m` no **mesmo split** para mostrar,
empiricamente, que **aumentar o modelo não resolve** — em 85 imagens, modelos
maiores tendem a **overfitar** e não melhoram a generalização (e ficam mais lentos).
Por isso `s` (ou no máximo `m`) é a escolha certa, não um XL/yolo26.

In [ ]:
import time

cmp_train, cmp_val = stratified_split(images, val_frac=0.2, seed=SEED)
cmp_yaml = build_yolo_dataset(cmp_train, cmp_val, "ds_modelcmp")

cmp_rows = []
for mdl in ["yolo11n.pt", "yolo11s.pt", "yolo11m.pt"]:
    t0 = time.time()
    mm = YOLO(mdl)
    mm.train(data=str(cmp_yaml), epochs=60, patience=20, imgsz=IMGSZ, batch=BATCH,
             seed=SEED, optimizer="auto", cos_lr=True, cache=True,
             project="runs_torneiras", name=f"cmp_{mdl[:-3]}", exist_ok=True,
             verbose=False, plots=False, **AUG)
    rr = mm.val(data=str(cmp_yaml), split="val", verbose=False)
    cmp_rows.append({"modelo": mdl, "mAP50": float(rr.box.map50),
                     "mAP50-95": float(rr.box.map), "min": (time.time()-t0)/60})

cmp_df = pd.DataFrame(cmp_rows)
print(cmp_df.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
print("\nConclusão: subir o tamanho do modelo não compensa nesse dataset pequeno.")


## 10. Inferência no conjunto de validação (melhor fold)

Comparação visual: **anotação real** (esquerda) × **predição do modelo** (direita).

In [ ]:
best = YOLO(str(BEST_WEIGHTS))
val_image_files = sorted((Path(best_fold["yaml"]).parent / "images" / "val").glob("*.*"))
sample_val = val_image_files[:min(5, len(val_image_files))]

plt.figure(figsize=(12, 4*len(sample_val)))
for i, img in enumerate(sample_val):
    pred = best.predict(str(img), imgsz=IMGSZ, conf=0.25, verbose=False)[0]
    pred_img = cv2.cvtColor(pred.plot(), cv2.COLOR_BGR2RGB)
    plt.subplot(len(sample_val), 2, 2*i+1)
    plt.imshow(draw_boxes(img)); plt.axis("off"); plt.title(f"Real — {img.name[:20]}", fontsize=9)
    plt.subplot(len(sample_val), 2, 2*i+2)
    plt.imshow(pred_img); plt.axis("off"); plt.title("Predição do modelo", fontsize=9)
plt.tight_layout(); plt.show()


## 11. Teste em imagens NOVAS  📸

Item 4.4 do desafio: tire **3 a 5 fotos novas** (abertas e fechadas) e faça upload
para testar a generalização em imagens que o modelo **nunca viu**.

In [ ]:
CONF = 0.25   # confiança mínima da detecção (ajuste se houver muitos FP/FN)
IOU  = 0.50

new_images = []
try:
    from google.colab import files
    print("Selecione de 3 a 5 fotos novas de torneiras...")
    uploaded = files.upload()
    new_images = [Path(name) for name in uploaded.keys()]
except Exception:
    folder = Path("novas_imagens")
    if folder.exists():
        new_images = sorted([p for p in folder.glob("*.*")
                             if p.suffix.lower() in {".jpg", ".jpeg", ".png"}])
    print(f"{len(new_images)} imagem(ns) encontradas em 'novas_imagens/'.")


In [ ]:
if new_images:
    plt.figure(figsize=(7, 6*len(new_images)))
    for i, img in enumerate(new_images):
        res = best.predict(str(img), imgsz=IMGSZ, conf=CONF, iou=IOU, verbose=False)[0]
        out = cv2.cvtColor(res.plot(), cv2.COLOR_BGR2RGB)
        dets = [f"{CLASS_NAMES[int(b.cls)]} ({float(b.conf):.2f})" for b in res.boxes]
        plt.subplot(len(new_images), 1, i+1)
        plt.imshow(out); plt.axis("off")
        plt.title(f"{img.name}  →  " + (", ".join(dets) if dets else "nada detectado"),
                  fontsize=10)
    plt.tight_layout(); plt.show()
else:
    print("Nenhuma imagem nova carregada — rode a célula anterior para fazer upload.")


## 12. Conclusões e análise

### Os resultados iniciais (mAP50 ≈ 0.38) faziam sentido?
**Em parte.** Com 85 imagens, pista de estado sutil e só 17 imagens de validação
(métrica ruidosa), números modestos são esperados — mas 0.38 estava **abaixo do
que a tarefa rende**. O sintoma "melhor época = 4, depois nada" denunciava
**augmentation agressiva demais** (`mosaic`/`mixup`) destruindo a pista de estado.
Após desligá-las e avaliar com 5-fold, o resultado fica mais alto e, sobretudo,
**confiável** (média ± desvio).

### E rodar um YOLO XL / yolo26 xlarge?
**Não ajudaria** — pelo contrário. Em 85 imagens, mais parâmetros = **mais
overfitting** e treino mais lento, sem ganho de generalização (Seção 9 mostra isso).
O gargalo é **quantidade/qualidade de dados e configuração**, não a capacidade do
modelo. `yolo11s` (ou no máximo `m`) é a escolha certa.

### Sobre o pré-processamento do Roboflow
- ❌ **"Isolate Objects"**: é técnica de *classificação* (recorta cada objeto numa
  imagem). Para **detecção** quebra o modelo — remove o contexto e faz a box virar a
  imagem inteira; em fotos reais (cena completa) a detecção falha. **Não usar.**
- ❌ **"Grayscale 100%"**: descarta cor (água/metal) e o backbone COCO é RGB. **Não
  usar** como pré-processamento fixo. *(Grayscale como augmentation de 15% é ok.)*
- ⚠️ **"Stretch 640"** distorce proporção → preferir o *letterbox* padrão do YOLO.
- ⚠️ **"90° rotate"** é irreal para torneira (não fica de lado) → remover. Brilho/flip/
  rotação ±15° leves são bons.

### Como melhorar ainda mais (maior alavanca primeiro)
1. **Mais dados** (alvo 200–300 imgs) com variação de ângulo/luz/fundo — maior ganho.
2. **Anotação consistente** + evitar imagens ambíguas (torneira aberta sem água visível
   parece fechada).
3. **Fotos que mostrem bem o registro/alavanca** (a pista de estado).
4. Ajustar `CONF`/`IOU` na inferência conforme o erro observado.

### 🏭 Paralelo industrial (resumo)
O mesmo pipeline — detectar **aberto/fechado** de um objeto — aplica-se ao
**monitoramento de válvulas e registros industriais**, onde identificar
automaticamente o estado de um atuador previne vazamentos, desperdício e falhas.
